# BIOMASS L1C Point Extraction: SLC Windows, Multilooked Intensity, Closure Phase and Baselines

For every station with BIOMASS L1C coverage inside its measurement period (from *Biomass L1C Point Availability.ipynb*), this notebook:

1. finds the station's position in the slant-range geometry of the L1C stack (all products of a stack share the coregistration-primary grid);
2. streams an SLC window covering about **200 m × 200 m on the ground** (azimuth × ground range) around that pixel for every acquisition and polarisation (COG range reads, not full downloads). The number of lines and samples is derived from the azimuth pixel spacing and from the slant-range spacing projected on the ground at the station's incidence angle;
3. computes the **multilooked intensity** (β⁰ and σ⁰), the **sample coherence matrix**, the **interferometric phases** and the **closure phase** of every triplet;
4. derives the **acquisition geometry and baselines** at the station from the orbit state vectors (perpendicular/parallel/along-track baseline, slant range, incidence and look angle, vertical wavenumber, heading) and reports the L1C LUT values (`waveNumbers`, `incidenceAngle`, annotated `normalBaseline`) next to them as a cross-check.

Product structure follows the *BIOMASS L1a/b/c Products Format Specification* (BIO-BPS-L1PFD-ARE-010076, issue 1.6.1, Sec. 6).

> **Validated on ROMPS/Baytik** (6-acquisition S1 stack, Nov–Dec 2025): the LUT- and orbit-based pixel positions agree within 0.004 lines / 0.35 samples; orbit B⊥ is within ~1 % of the annotated `normalBaseline`; `kz_orbit` matches the `waveNumbers` LUT; with the default 200 m × 200 m window (29 lines × 5 samples = 195 m × 221 m, 145 looks), applying the flattening phase with the "+" convention raises the mean coherence (HH 0.17 → 0.33), the opposite sign lowers it. Other stacks have not been run yet: check `locations` (Sec. 6.1) for every new station.
>
> Runtime is ~35 s per acquisition (mostly LUT reads).

### 1. Packages and Libraries

In [ ]:
%pip install -q aiohttp requests h5netcdf h5py

from scipy.interpolate import CubicHermiteSpline
from scipy.ndimage import map_coordinates
from scipy.optimize import brentq
import xml.etree.ElementTree as ET
from rasterio.windows import Window
from tqdm import tqdm
from itertools import combinations
import matplotlib.pyplot as plt
import geopandas as gpd
import rasterio as rio
import xarray as xr
import pandas as pd
import numpy as np
import traceback
import requests
import pathlib
import fsspec
import time

### 2. Parameters

- `OUTPUT_DIR`: folder where all outputs (tables, SLC windows) and the download cache are written, e.g. `/home/jovyan/test_data`. It is created if missing.
- `AVAILABILITY_DIR`: output folder of *Biomass L1C Point Availability.ipynb* (`points_l1c_availability.geojson`, `points_l1c_products.csv`); by default `OUTPUT_DIR/l1c_availability`, where that notebook writes when both use the same `OUTPUT_DIR`.
- `WINDOW_AZ_M` / `WINDOW_GR_M`: multilook window extent on the ground, in azimuth and ground range [m]. Converted per stack to the nearest odd number of lines (azimuth spacing) and samples (slant-range spacing / sin θ at the station), so the station stays at the centre pixel.
- `POINT_IDS`: stations to process. Start with one; set to `None` for all stations that have an in-period stack.
- **Acquisitions:** for each station, every stack that has at least one product with matched soil moisture (`REQUIRE_SM = True`, from the availability notebook) — or, with `REQUIRE_SM = False`, at least one product inside the station's measurement period — is processed with **all** its acquisitions. Frames are kept apart: if a station falls in the overlap of two frames of the same stack, both are processed separately.
- `POLARISATIONS`: `None` = all bands in the product. L1C stacks may store the averaged cross-pol as a single `XX` band instead of `HV`/`VH`.
- `APPLY_FLATTENING` / `APPLY_SKP`: the phase state you want for the interferometric analysis. The notebook reads which corrections were already applied in each product (`skpPhaseCorrectionFlag`, `skpPhaseCorrectionFlatteningOnlyFlag`) and applies or removes only the difference, using the L1C "+" convention `SLC·exp(+j(Δφ_flat + φ_skp))`. Removing the topographic (flattening) phase before multilooking is important: with BIOMASS tomographic baselines it varies by several radians inside the window (up to 5.8 rad within only 5×5 pixels at Baytik), which would otherwise bias coherence and closure phase. The SKP screen is a smooth ground-phase calibration; leaving it out keeps the ground phase signal.
- `DOWNLOAD_LUT`: `False` (default) reads only the needed LUT variables over HTTP (~3x less data); `True` downloads each full `lut.nc` (~70 MB) into `CACHE_DIR`, useful for re-runs.

In [ ]:
OUTPUT_DIR = pathlib.Path("/home/jovyan/test_data")   # <- where all outputs are written
AVAILABILITY_DIR = OUTPUT_DIR / "l1c_availability"    # outputs of Biomass L1C Point Availability.ipynb
POINT_IDS = ["ROMPS/Baytik"]      # None = all selected stations
REQUIRE_SM = True                  # only stacks with soil moisture at >= 1 acquisition
WINDOW_AZ_M = 200.0                # multilook window extent in azimuth [m]
WINDOW_GR_M = 200.0                # multilook window extent in ground range [m]
POLARISATIONS = None               # None = all, or e.g. ["HH", "VV"]
APPLY_FLATTENING = True
APPLY_SKP = False
DOWNLOAD_LUT = False
TOKEN_FILE = "../../token.txt"     # text file containing your MAAP offline token (repo root)
CACHE_DIR = OUTPUT_DIR / "cache"   # downloaded annotations, orbits (and LUTs if DOWNLOAD_LUT)

CATALOG_URL = "https://catalog.maap.eo.esa.int/catalogue/"
COLLECTION = "BiomassLevel1c"

### 3. Authentication

Data access requires a token generated with an authorized ESA account (EO Sign in). Generate your personal offline token [here](https://portal.maap.eo.esa.int/ini/services/auth/token/index.php) (valid for 90 days) and paste it into the file `TOKEN_FILE`. Short-lived access tokens are refreshed automatically.

In [ ]:
# This is permanently valid, you don't need to change the two lines below.
CLIENT_ID="offline-token"
CLIENT_SECRET="p1eL7uonXs6MDxtGbgKdPVRAmnGxHpVE"

IAM_URL = "https://iam.maap.eo.esa.int/realms/esa-maap/protocol/openid-connect/token"

if not pathlib.Path(TOKEN_FILE).exists():
    raise RuntimeError(f"No token file found at {pathlib.Path(TOKEN_FILE).resolve()} !")
OFFLINE_TOKEN = pathlib.Path(TOKEN_FILE).read_text().strip().replace("\n", "")

_token = {"value": None, "expires": 0.0}


def get_token():
    """Return a valid access token, refreshing it one minute before it expires."""
    if _token["value"] is None or time.time() > _token["expires"] - 60:
        response = requests.post(IAM_URL, data={
            "client_id": CLIENT_ID,
            "client_secret": CLIENT_SECRET,
            "grant_type": "refresh_token",
            "refresh_token": OFFLINE_TOKEN,
            "scope": "offline_access openid",
        })
        response.raise_for_status()
        response_json = response.json()
        if not response_json.get("access_token"):
            raise RuntimeError("Failed to retrieve access token from IAM response")
        _token["value"] = response_json["access_token"]
        _token["expires"] = time.time() + response_json.get("expires_in", 300)
    return _token["value"]


def auth_headers():
    return {"Authorization": f"Bearer {get_token()}"}


get_token()
print("Access token retrieved successfully.")

### 4. Helper functions

#### 4.1. Remote access: items, annotations, LUTs and COG windows

Product files are located from the item assets by their folder inside the product:
- `annotation_coregistered/` holds this acquisition's main annotation, its LUT (`*_lut.nc`) and its own orbit (`navigation/*_orb.xml`);
- `annotation_primary/` holds the coregistration primary's annotation and orbit.

SLC windows are read with GDAL range requests (`/vsicurl/` with a bearer header), so only the few kB around the station are transferred.

In [ ]:
def get_item(product_id):
    r = requests.get(f"{CATALOG_URL}collections/{COLLECTION}/items/{product_id}")
    r.raise_for_status()
    return r.json()


def asset_href(item, folder, suffix):
    """Href of the asset located in `folder` (inside the product) whose name ends with `suffix`."""
    for a in item["assets"].values():
        h = a["href"]
        if (folder is None or f"/{folder}/" in h) and h.lower().endswith(suffix):
            return h
    raise KeyError(f"No asset in '{folder}' ending with '{suffix}' for {item['id']}")


def fetch_cached(url):
    path = CACHE_DIR / url.split("/data/", 1)[-1]
    if not path.exists():
        path.parent.mkdir(parents=True, exist_ok=True)
        with requests.get(url, headers=auth_headers(), stream=True) as r:
            r.raise_for_status()
            tmp = path.with_suffix(path.suffix + ".part")
            with open(tmp, "wb") as f:
                for chunk in r.iter_content(chunk_size=2**20):
                    f.write(chunk)
            tmp.rename(path)
    return path


def read_xml(url):
    return ET.parse(fetch_cached(url)).getroot()


LUT_VARS = {
    "coregistration": ["waveNumbers", "flatteningPhaseScreen"],
    "skpPhaseCalibration": ["skpCalibrationPhaseScreen"],
    "radiometry": ["sigmaNought"],
    "geometry": ["incidenceAngle", "height", "rangeTerrainSlope", "azimuthTerrainSlope"],
}


def read_lut(url, groups, isl=slice(None), jsl=slice(None)):
    """Read LUT variables {group: [names]} (optionally a sub-window) plus the LUT axes and attributes."""
    if DOWNLOAD_LUT:
        src = fetch_cached(url)
    else:
        src = fsspec.filesystem("https", headers=auth_headers()).open(
            url, "rb", block_size=2**22, cache_type="blockcache")
    opts = dict(engine="h5netcdf", decode_times=False, decode_timedelta=False)
    out = {}
    try:
        with xr.open_dataset(src, **opts) as root:
            out["az_axis"] = root["relativeAzimuthTime"].values.astype(float)
            out["rg_axis"] = root["slantRangeTime"].values.astype(float)
            out["attrs"] = dict(root.attrs)
        for group, names in groups.items():
            try:
                with xr.open_dataset(src, group=group, **opts) as ds:
                    for n in names:
                        if n in ds:
                            da = ds[n].isel(relativeAzimuthTime=isl, slantRangeTime=jsl)
                            out[n] = da.values.astype(float)
            except (OSError, KeyError, ValueError):
                pass  # group not present in this product
    finally:
        if hasattr(src, "close"):
            src.close()
    return out


def band_polarisations(src):
    seq = src.tags().get("PolarisationsSequence")
    if seq:
        return seq.replace("/", "").split()
    return [src.tags(b).get("POLARIMETRIC_INTERP", f"B{b}") for b in src.indexes]


def read_cog_window(url, window):
    """Read all bands of a COG inside `window` with HTTP range requests."""
    with rio.Env(GDAL_HTTP_HEADERS=f"Authorization: Bearer {get_token()}",
                 GDAL_DISABLE_READDIR_ON_OPEN="EMPTY_DIR",
                 CPL_VSIL_CURL_ALLOWED_EXTENSIONS=".tiff,.tif",
                 GDAL_HTTP_MAX_RETRY="3"):
        with rio.open(url) as src:
            if (window.row_off < 0 or window.col_off < 0
                    or window.row_off + window.height > src.height
                    or window.col_off + window.width > src.width):
                raise ValueError(f"Window {window} outside raster {src.height}x{src.width}")
            return src.read(window=window).astype(np.float64), band_polarisations(src)

#### 4.2. Annotation parsing

In [ ]:
def to_dt(s):
    return np.datetime64(s.strip().replace("UTC=", "").rstrip("Z"), "ns")


def seconds(td):
    return td / np.timedelta64(1, "s")


def _find(root, path):
    el = root.find(path)
    return None if el is None or el.text is None else el.text.strip()


def _bool(s, default=False):
    return default if s is None else s.strip().lower() == "true"


def parse_annotation(root):
    """Fields of an L1C (coregistered) main annotation needed here."""
    sar = root.find(".//{*}sarImage")
    g = lambda n: _find(sar, f"{{*}}{n}")
    pol_el = root.find(".//{*}acquisitionInformation/{*}polarisationList")
    pols = [c.text.strip() for c in pol_el] if pol_el is not None and len(pol_el) else \
           (pol_el.text.split() if pol_el is not None and pol_el.text else [])
    corr = _bool(_find(root, ".//{*}staProcessingParameters/{*}skpPhaseCorrectionFlag"))
    flat_only = _bool(_find(root, ".//{*}staProcessingParameters/{*}skpPhaseCorrectionFlatteningOnlyFlag"))
    nb = _find(root, ".//{*}staCoregistrationParameters/{*}normalBaseline")
    fc = _find(root, ".//{*}radarCarrierFrequency")
    return {
        "first_line": to_dt(g("firstLineAzimuthTime")),
        "first_sample_rt": float(g("firstSampleSlantRangeTime")),
        "dt_az": float(g("azimuthTimeInterval")),
        "dt_rg": float(g("rangeTimeInterval")),
        "rg_spacing": float(g("rangePixelSpacing")),                # slant range [m]
        "az_spacing": float(g("azimuthPixelSpacing")),              # [m]
        "n_lines": int(g("numberOfLines")),
        "n_samples": int(g("numberOfSamples")),
        "no_data": float(g("noDataValue")) if g("noDataValue") else np.nan,
        "pixel_quantity": g("pixelQuantity") or "Beta-Nought",
        "polarisations": [p.replace("/", "") for p in pols],
        "carrier_freq": float(fc) if fc else 435e6,
        "primary_image": _find(root, ".//{*}staCoregistrationParameters/{*}primaryImage"),
        "normal_baseline_annot": float(nb) if nb else np.nan,
        "flat_applied": corr,
        "skp_applied": corr and not flat_only,
    }

#### 4.3. Orbit, geodesy and acquisition geometry

Orbits are read from the EOF orbit files (`List_of_OSVs/OSV`) and interpolated with a cubic Hermite spline (positions + velocities). For a target **P** (station position at the DEM height from the LUT), the zero-Doppler time solves (**P** − **S**(t))·**V**(t) = 0.

With **S**ₖ (this acquisition) and **S**₀ (coregistration primary) at their zero-Doppler times, unit line of sight **l̂** = (**P** − **S**₀)/R₀ and unit velocity **v̂**:
- along-track baseline B‖ₐ = **B**·**v̂**, parallel baseline B∥ = **B**·**l̂**;
- perpendicular baseline B⊥ = **B**·**ê**⊥ with **ê**⊥ = unit(**v̂** × **l̂**), oriented away from the Earth (positive = secondary above the primary's line of sight);
- vertical wavenumber k_z = −4π B⊥ / (λ R₀ sin θ), with θ the local incidence angle on the ellipsoid. The minus sign reproduces the convention of the L1C `waveNumbers` LUT.

With this convention B⊥ has the same sign as the annotated `normalBaseline` (scene average); compare `b_perp_m` with `normal_baseline_annot_m` and `kz_orbit` with `kz_lut`.

In [ ]:
C_LIGHT = 299792458.0
WGS84_A, WGS84_F = 6378137.0, 1 / 298.257223563
WGS84_E2 = WGS84_F * (2 - WGS84_F)


def geodetic_to_ecef(lat, lon, h):
    la, lo = np.radians(lat), np.radians(lon)
    n = WGS84_A / np.sqrt(1 - WGS84_E2 * np.sin(la) ** 2)
    return np.array([(n + h) * np.cos(la) * np.cos(lo),
                     (n + h) * np.cos(la) * np.sin(lo),
                     (n * (1 - WGS84_E2) + h) * np.sin(la)])


def enu_basis(lat, lon):
    la, lo = np.radians(lat), np.radians(lon)
    e = np.array([-np.sin(lo), np.cos(lo), 0.0])
    n = np.array([-np.sin(la) * np.cos(lo), -np.sin(la) * np.sin(lo), np.cos(la)])
    u = np.array([np.cos(la) * np.cos(lo), np.cos(la) * np.sin(lo), np.sin(la)])
    return e, n, u


class Orbit:
    def __init__(self, root):
        osvs = root.findall(".//{*}OSV")
        if len(osvs) < 4:
            raise ValueError("Orbit file has fewer than 4 state vectors")
        times = np.array([to_dt(o.find("{*}UTC").text) for o in osvs])
        pos = np.array([[float(o.find(f"{{*}}{k}").text) for k in ("X", "Y", "Z")] for o in osvs])
        vel = np.array([[float(o.find(f"{{*}}{k}").text) for k in ("VX", "VY", "VZ")] for o in osvs])
        self.epoch = times.min()
        t = seconds(times - self.epoch)
        t, idx = np.unique(t, return_index=True)
        self.t = t
        self.pos = CubicHermiteSpline(t, pos[idx], vel[idx], axis=0)
        self.vel = self.pos.derivative()

    def state(self, ts):
        return self.pos(ts), self.vel(ts)

    def zero_doppler(self, P):
        """Zero-Doppler time (s since epoch) of target P; the root with the shortest range is kept."""
        f = lambda ts: np.dot(P - self.pos(ts), self.vel(ts))
        fv = np.array([f(ts) for ts in self.t])
        k = np.where(np.sign(fv[:-1]) != np.sign(fv[1:]))[0]
        if not len(k):
            raise ValueError("Target is not imaged within the orbit time span")
        roots = [brentq(f, self.t[i], self.t[i + 1], xtol=1e-10) for i in k]
        return min(roots, key=lambda ts: np.linalg.norm(P - self.pos(ts)))

    def to_datetime(self, ts):
        return self.epoch + np.timedelta64(int(round(ts * 1e9)), "ns")


def acquisition_geometry(P, lat, lon, orbit, orbit_ref, wavelength):
    """Geometry of one acquisition at target P, with baselines w.r.t. the coregistration primary orbit."""
    t = orbit.zero_doppler(P)
    S, _ = orbit.state(t)
    t0 = orbit_ref.zero_doppler(P)
    S0, V0 = orbit_ref.state(t0)

    los = P - S0
    r0 = np.linalg.norm(los)
    l_hat = los / r0
    v_hat = V0 / np.linalg.norm(V0)
    e_perp = np.cross(v_hat, l_hat)
    e_perp /= np.linalg.norm(e_perp)
    if np.dot(e_perp, S0) < 0:
        e_perp = -e_perp

    B = S - S0
    e, n, u = enu_basis(lat, lon)
    inc = np.arccos(np.dot(-l_hat, u))
    b_perp = np.dot(B, e_perp)
    return {
        "zero_doppler_time": orbit.to_datetime(t),
        "slant_range_m": np.linalg.norm(P - S),
        "primary_slant_range_m": r0,
        "b_perp_m": b_perp,
        "b_par_m": np.dot(B, l_hat),
        "b_along_m": np.dot(B, v_hat),
        "incidence_deg_orbit": np.degrees(inc),
        "look_deg": np.degrees(np.arccos(np.dot(l_hat, -S0 / np.linalg.norm(S0)))),
        "kz_orbit": -4 * np.pi * b_perp / (wavelength * r0 * np.sin(inc)),  # sign of the L1C waveNumbers LUT
        "heading_deg": np.degrees(np.arctan2(np.dot(v_hat, e), np.dot(v_hat, n))) % 360,
        "_t0": orbit_ref.to_datetime(t0),
    }

#### 4.4. Locating the station in the slant-range grid

The geometry LUT gives latitude/longitude (DEM-based, ~90 m posting) on the `(relativeAzimuthTime, slantRangeTime)` grid. The nearest LUT node is refined to sub-node precision with a local affine fit of east/north offsets over the surrounding 5×5 nodes, then converted to SLC line/sample with the primary grid annotation:

line = (t_az − firstLineAzimuthTime) / azimuthTimeInterval, sample = (t_rg − firstSampleSlantRangeTime) / rangeTimeInterval.

The same is done independently with the primary orbit (zero-Doppler time and two-way range of the station). The difference (`d_line_orbit`, `d_sample_orbit`) should be well below one pixel; larger values point to a datum/height or timing problem.

In [ ]:
def locate_in_lut(lat, lon, lut_lat, lut_lon, half=2):
    """Fractional LUT indices (fi, fj) of (lat, lon), and distance of the nearest LUT node [m]."""
    R = 6371000.0
    dlon = (lut_lon - lon + 180) % 360 - 180
    E = np.radians(dlon) * R * np.cos(np.radians(lat))
    N = np.radians(lut_lat - lat) * R
    d = np.hypot(E, N)
    i, j = np.unravel_index(np.nanargmin(d), d.shape)
    si = slice(max(i - half, 0), i + half + 1)
    sj = slice(max(j - half, 0), j + half + 1)
    DI, DJ = np.meshgrid(np.arange(si.start, si.start + E[si, sj].shape[0]) - i,
                         np.arange(sj.start, sj.start + E[si, sj].shape[1]) - j, indexing="ij")
    ok = np.isfinite(E[si, sj]) & np.isfinite(N[si, sj])
    A = np.c_[np.ones(ok.sum()), DI[ok], DJ[ok]]
    cE = np.linalg.lstsq(A, E[si, sj][ok], rcond=None)[0]
    cN = np.linalg.lstsq(A, N[si, sj][ok], rcond=None)[0]
    di, dj = np.linalg.solve([[cE[1], cE[2]], [cN[1], cN[2]]], [-cE[0], -cN[0]])
    on_edge = i in (0, d.shape[0] - 1) or j in (0, d.shape[1] - 1)
    return i + di, j + dj, float(d[i, j]), bool(on_edge)


def axis_at(axis, f):
    """Axis value at fractional index f (linear, extrapolating at the ends)."""
    idx = np.arange(len(axis))
    return np.interp(f, idx, axis, left=axis[0] + (f - 0) * (axis[1] - axis[0]),
                     right=axis[-1] + (f - idx[-1]) * (axis[-1] - axis[-2]))


def index_of(axis, value):
    """Fractional index of `value` on a monotonic axis."""
    return np.interp(value, axis, np.arange(len(axis)))


def interp_lut(arr, fi, fj):
    """Bilinear interpolation of a LUT (sub)array at fractional indices."""
    if arr is None:
        return np.full(np.shape(fi), np.nan)
    return map_coordinates(arr, [fi, fj], order=1, mode="nearest")

#### 4.5. Multilooking, coherence and closure phase

For one polarisation and N acquisitions (sorted in time), the sample covariance over the ~200 m × 200 m window is C_ij = ⟨s_i s_j*⟩ using pixels valid in all acquisitions, the coherence γ_ij = C_ij / √(C_ii C_jj), and the closure phase of a triplet i<j<k is

Φ_ijk = arg(C_ij · C_jk · C_ik*).

Any phase term that is constant over the window and depends on a single acquisition (orbit errors, atmosphere, residual calibration screens) cancels in Φ_ijk.

An acquisition whose required phase screen (flattening, or SKP if requested) is invalid anywhere in the window — this happens near swath edges, outside the area where the stack LUTs are defined — gets `phase_ok = False`: its intensity is kept, but it is left out of the coherence matrix, pairs and triplets. `i`, `j`, `k` always refer to the position in the full, time-sorted stack; `consecutive` means consecutive among the acquisitions with valid phase.

In [ ]:
def coherence_matrix(stack):
    """stack: (n_acq, W, W) complex, NaN = invalid. Returns C, coherence, number of looks."""
    X = stack.reshape(stack.shape[0], -1)
    valid = np.all(np.isfinite(X), axis=0)
    X = X[:, valid]
    if X.shape[1] == 0:
        n = stack.shape[0]
        return np.full((n, n), np.nan + 0j), np.full((n, n), np.nan + 0j), 0
    C = X @ X.conj().T / X.shape[1]
    p = np.real(np.diag(C))
    return C, C / np.sqrt(np.outer(p, p)), int(valid.sum())


def closure_phase(C, i, j, k):
    return float(np.angle(C[i, j] * C[j, k] * np.conj(C[i, k])))

#### 4.6. Process one stack at one station

In [ ]:
def process_stack(point_id, lat, lon, product_ids):
    items = sorted((get_item(p) for p in product_ids), key=lambda it: it["properties"]["start_datetime"])
    anns = [parse_annotation(read_xml(asset_href(it, "annotation_coregistered", "_annot.xml"))) for it in items]
    grid = anns[0]
    for it, a in zip(items, anns):
        if (a["n_lines"], a["n_samples"], a["first_sample_rt"], a["dt_az"]) !=                 (grid["n_lines"], grid["n_samples"], grid["first_sample_rt"], grid["dt_az"]):
            print(f"  WARNING: {it['id']} is not on the same grid as {items[0]['id']}")
    primary_first_line = parse_annotation(
        read_xml(asset_href(items[0], "annotation_primary", "_annot.xml")))["first_line"]

    # --- 1. station position in the slant-range grid (geometry LUT) ---
    lut_url0 = asset_href(items[0], "annotation_coregistered", "_lut.nc")
    geo = read_lut(lut_url0, {"geometry": ["latitude", "longitude", "height", "incidenceAngle"]})
    fi, fj, node_dist, on_edge = locate_in_lut(lat, lon, geo["latitude"], geo["longitude"])
    if on_edge and node_dist > 200:
        raise ValueError(f"Station is outside the LUT grid (nearest node {node_dist:.0f} m away)")
    ref_attr = geo["attrs"].get("referenceAzimuthTime")
    lut_ref = to_dt(str(ref_attr)) if ref_attr is not None else grid["first_line"]
    lut_offset = seconds(lut_ref - grid["first_line"])         # LUT azimuth reference vs first line [s]
    az_axis, rg_axis = geo["az_axis"], geo["rg_axis"]
    t_az = axis_at(az_axis, fi) + lut_offset                   # [s] since first line
    t_rg = axis_at(rg_axis, fj)
    line = t_az / grid["dt_az"]
    sample = (t_rg - grid["first_sample_rt"]) / grid["dt_rg"]
    height = float(interp_lut(geo["height"], np.array([fi]), np.array([fj]))[0])
    P = geodetic_to_ecef(lat, lon, height)

    # window size: ~WINDOW_AZ_M x WINDOW_GR_M on the ground, odd number of lines/samples
    inc_point = float(interp_lut(geo["incidenceAngle"], np.array([fi]), np.array([fj]))[0])
    inc_source = "lut"
    if not np.isfinite(inc_point):
        orb0 = Orbit(read_xml(asset_href(items[0], "annotation_primary/navigation", "_orb.xml")))
        S0, _ = orb0.state(orb0.zero_doppler(P))
        inc_point = float(np.degrees(np.arccos(np.dot((S0 - P) / np.linalg.norm(S0 - P), enu_basis(lat, lon)[2]))))
        inc_source = "orbit"
        print(f"  NOTE: incidence LUT invalid at the station, window sized with the orbit value {inc_point:.2f} deg")
    gr_spacing = grid["rg_spacing"] / np.sin(np.radians(inc_point))
    nearest_odd = lambda x: max(1, 2 * int(round((x - 1) / 2)) + 1)
    n_az, n_rg = nearest_odd(WINDOW_AZ_M / grid["az_spacing"]), nearest_odd(WINDOW_GR_M / gr_spacing)
    ha, hr = n_az // 2, n_rg // 2
    r0, c0 = int(round(line)), int(round(sample))
    window = Window(c0 - hr, r0 - ha, n_rg, n_az)
    rows = np.arange(r0 - ha, r0 + ha + 1)
    cols = np.arange(c0 - hr, c0 + hr + 1)
    FI, FJ = np.meshgrid(index_of(az_axis, rows * grid["dt_az"] - lut_offset),
                         index_of(rg_axis, grid["first_sample_rt"] + cols * grid["dt_rg"]), indexing="ij")
    i0, i1 = max(int(np.floor(FI.min())) - 1, 0), int(np.ceil(FI.max())) + 2
    j0, j1 = max(int(np.floor(FJ.min())) - 1, 0), int(np.ceil(FJ.max())) + 2

    location = {"point_id": point_id, "lut_fi": fi, "lut_fj": fj, "nearest_lut_node_m": node_dist,
                "line": line, "sample": sample, "window_row0": r0 - ha, "window_col0": c0 - hr,
                "window_lines": n_az, "window_samples": n_rg,
                "window_az_m": n_az * grid["az_spacing"], "window_gr_m": n_rg * gr_spacing,
                "az_spacing_m": grid["az_spacing"], "ground_range_spacing_m": gr_spacing,
                "incidence_deg": inc_point, "incidence_source": inc_source, "dem_height_m": height}

    acq_rows, slcs, pols_ref, phase_oks = [], [], None, []
    raw_slcs, flat_screens, skp_screens = [], [], []
    for it, ann in zip(items, anns):
        # --- 2. SLC window ---
        amp, pols = read_cog_window(asset_href(it, "measurement", "_i_abs.tiff"), window)
        pha, _ = read_cog_window(asset_href(it, "measurement", "_i_phase.tiff"), window)
        keep = [k for k, p in enumerate(pols) if POLARISATIONS is None or p in POLARISATIONS]
        pols = [pols[k] for k in keep]
        if pols_ref is None:
            pols_ref = pols
        elif pols != pols_ref:
            raise ValueError(f"Polarisations differ within the stack: {pols} vs {pols_ref}")
        amp, pha = amp[keep], pha[keep]
        invalid = ~np.isfinite(amp) | (amp == ann["no_data"])
        slc = np.where(invalid, np.nan, amp * np.exp(1j * pha))

        # --- 3. LUT values at the window pixels ---
        lut = read_lut(asset_href(it, "annotation_coregistered", "_lut.nc"), LUT_VARS,
                       slice(i0, i1), slice(j0, j1))
        sub = lambda n: interp_lut(lut.get(n), FI - i0, FJ - j0)
        flat, skp = sub("flatteningPhaseScreen"), sub("skpCalibrationPhaseScreen")
        need_flat = int(APPLY_FLATTENING) - int(ann["flat_applied"])
        need_skp = int(APPLY_SKP) - int(ann["skp_applied"])
        missing_flat = need_flat != 0 and np.isnan(flat).any()
        missing_skp = need_skp != 0 and np.isnan(skp).any()
        phase_ok = not (missing_flat or missing_skp)
        phase_oks.append(phase_ok)
        if not phase_ok:
            print(f"  WARNING: {it['id']}: {'flattening' if missing_flat else 'SKP'} LUT invalid in the window "
                  f"-> excluded from coherence / closure phase (intensity kept)")
        extra = need_flat * np.nan_to_num(flat) + need_skp * np.nan_to_num(skp)
        raw_slcs.append(slc)
        flat_screens.append(flat)
        skp_screens.append(skp)
        slc = slc * np.exp(1j * extra)

        power = np.abs(slc) ** 2
        if "sigma" in ann["pixel_quantity"].lower():
            beta0, sigma0 = np.full_like(power, np.nan), power
        else:
            beta0, sigma0 = power, power * sub("sigmaNought") ** 2
        slcs.append(slc)

        # --- 4. orbit geometry ---
        orb = Orbit(read_xml(asset_href(it, "annotation_coregistered/navigation", "_orb.xml")))
        orb_ref = Orbit(read_xml(asset_href(it, "annotation_primary/navigation", "_orb.xml")))
        geom = acquisition_geometry(P, lat, lon, orb, orb_ref, C_LIGHT / ann["carrier_freq"])
        t0 = geom.pop("_t0")
        if "line_orbit" not in location:
            location["line_orbit"] = seconds(t0 - primary_first_line) / grid["dt_az"]
            location["sample_orbit"] = (2 * geom["primary_slant_range_m"] / C_LIGHT
                                        - grid["first_sample_rt"]) / grid["dt_rg"]

        props = it["properties"]
        for p, pol in enumerate(pols):
            acq_rows.append({
                "point_id": point_id, "product_id": it["id"], "stack_id": props.get("eofeos:stack_id"),
                "primary_image": ann["primary_image"],
                "is_primary": bool(props.get("eofeos:is_coregistration_primary")),
                "datetime": pd.Timestamp(props["start_datetime"]), "pol": pol,
                "beta0": np.nanmean(beta0[p]), "sigma0": np.nanmean(sigma0[p]),
                "sigma0_db": 10 * np.log10(np.nanmean(sigma0[p])),
                "n_valid": int(np.isfinite(slc[p]).sum()), "phase_ok": phase_ok,
                "kz_lut": np.nanmean(sub("waveNumbers")),
                "incidence_deg_lut": np.nanmean(sub("incidenceAngle")),
                "range_slope_deg_lut": np.nanmean(sub("rangeTerrainSlope")),
                "azimuth_slope_deg_lut": np.nanmean(sub("azimuthTerrainSlope")),
                "flattening_phase_rad": np.nanmean(flat), "skp_phase_rad": np.nanmean(skp),
                "flat_applied_in_product": ann["flat_applied"], "skp_applied_in_product": ann["skp_applied"],
                "normal_baseline_annot_m": ann["normal_baseline_annot"],
                "orbit_state": props.get("sat:orbit_state"), "absolute_orbit": props.get("sat:absolute_orbit"),
                **geom,
            })

    location["d_line_orbit"] = location["line_orbit"] - location["line"]
    location["d_sample_orbit"] = location["sample_orbit"] - location["sample"]

    # --- 5. coherence matrix, pairs, closure phases ---
    cube = np.stack(slcs)                                     # (n_acq, n_pol, W, W)
    dates = [pd.Timestamp(it["properties"]["start_datetime"]).tz_convert(None) for it in items]
    acq = pd.DataFrame(acq_rows)
    bperp = acq.drop_duplicates("product_id").set_index("product_id")["b_perp_m"]
    pair_rows, trip_rows = [], []
    ok = [k for k, flag in enumerate(phase_oks) if flag]      # indices of acquisitions with valid phase
    for p, pol in enumerate(pols_ref):
        if len(ok) < 2:
            break
        C, coh, n_looks = coherence_matrix(cube[ok, p])
        for a, b in combinations(range(len(ok)), 2):
            i, j = ok[a], ok[b]
            pair_rows.append({
                "point_id": point_id, "pol": pol, "i": i, "j": j,
                "date_i": dates[i], "date_j": dates[j], "dt_days": (dates[j] - dates[i]).days,
                "consecutive": b == a + 1,
                "phase_rad": float(np.angle(C[a, b])), "coherence": float(np.abs(coh[a, b])),
                "d_b_perp_m": bperp[items[j]["id"]] - bperp[items[i]["id"]], "n_looks": n_looks,
            })
        for a, b, c in combinations(range(len(ok)), 3):
            i, j, k = ok[a], ok[b], ok[c]
            trip_rows.append({
                "point_id": point_id, "pol": pol, "i": i, "j": j, "k": k,
                "date_i": dates[i], "date_j": dates[j], "date_k": dates[k],
                "consecutive": b == a + 1 and c == b + 1,
                "closure_phase_rad": closure_phase(C, a, b, c), "n_looks": n_looks,
            })

    window_data = {"slc": cube.astype(np.complex64), "slc_raw": np.stack(raw_slcs).astype(np.complex64),
                   "flattening_phase": np.stack(flat_screens), "skp_phase": np.stack(skp_screens),
                   "flat_applied_in_product": np.array([a["flat_applied"] for a in anns]),
                   "skp_applied_in_product": np.array([a["skp_applied"] for a in anns]),
                   "phase_ok": np.array(phase_oks),
                   "pols": np.array(pols_ref),
                   "product_ids": np.array([it["id"] for it in items]),
                   "dates": np.array(dates, dtype="datetime64[ns]"), "rows": rows, "cols": cols}
    pair_cols = ["point_id", "pol", "i", "j", "date_i", "date_j", "dt_days", "consecutive", "phase_rad", "coherence",
                 "d_b_perp_m", "n_looks"]
    trip_cols = ["point_id", "pol", "i", "j", "k", "date_i", "date_j", "date_k", "consecutive",
                 "closure_phase_rad", "n_looks"]
    return (location, acq, pd.DataFrame(pair_rows, columns=pair_cols),
            pd.DataFrame(trip_rows, columns=trip_cols), window_data)

### 5. Select stations and stacks

In [ ]:
points = gpd.read_file(AVAILABILITY_DIR / "points_l1c_availability.geojson")
products = pd.read_csv(AVAILABILITY_DIR / "points_l1c_products.csv")
for c in ["in_period", "sm_available"]:
    products[c] = products[c].astype(str).str.lower().eq("true")
products["frame"] = products["product_id"].str.extract(r"_F(\d{3})_", expand=False)

selected = products["sm_available"] if REQUIRE_SM else products["in_period"]
selected_stacks = products.loc[selected, ["point_id", "stack_id"]].drop_duplicates()
jobs = products.merge(selected_stacks, on=["point_id", "stack_id"])
if POINT_IDS:
    jobs = jobs[jobs["point_id"].isin(POINT_IDS)]

stack_groups = jobs.groupby(["point_id", "stack_id", "frame"])["product_id"].apply(sorted)
print(f"{len(stack_groups)} stack(s) at {jobs['point_id'].nunique()} station(s), "
      f"{jobs['product_id'].nunique()} products in total.")
stack_groups.apply(len).rename("n_acquisitions").to_frame()

### 6. Extract SLC windows and compute intensity, coherence, closure phase and geometry

In [ ]:
(OUTPUT_DIR / "slc_windows").mkdir(parents=True, exist_ok=True)
coords = points.set_index("point_id").geometry

locations, acq_tables, pair_tables, trip_tables, errors = [], [], [], [], []
for (pid, stack_id, frame), pids in tqdm(stack_groups.items(), total=len(stack_groups)):
    print(f"{pid} | {stack_id} | F{frame} | {len(pids)} acquisitions")
    try:
        loc, acq, pairs, trips, win = process_stack(pid, coords[pid].y, coords[pid].x, pids)
        print(f"  window {loc['window_lines']} lines x {loc['window_samples']} samples "
              f"= {loc['window_az_m']:.0f} m x {loc['window_gr_m']:.0f} m")
    except Exception as e:
        traceback.print_exc()
        errors.append({"point_id": pid, "stack_id": stack_id, "frame": frame, "error": repr(e)})
        continue
    for df in (pairs, trips):
        df.insert(1, "stack_id", stack_id)
    for df in (acq, pairs, trips):
        df.insert(2, "frame", frame)
    locations.append({**loc, "stack_id": stack_id, "frame": frame})
    acq_tables.append(acq)
    pair_tables.append(pairs)
    trip_tables.append(trips)
    np.savez_compressed(OUTPUT_DIR / "slc_windows" / f"{pid.replace('/', '_')}_{stack_id}_F{frame}.npz", **win)

locations = pd.DataFrame(locations)
acquisitions = pd.concat(acq_tables, ignore_index=True) if acq_tables else pd.DataFrame()
pairs = pd.concat(pair_tables, ignore_index=True) if pair_tables else pd.DataFrame()
triplets = pd.concat(trip_tables, ignore_index=True) if trip_tables else pd.DataFrame()
if len(acquisitions):
    acquisitions = acquisitions.merge(products[["point_id", "product_id", "sm_surface", "sm_dt_min"]],
                                      on=["point_id", "product_id"], how="left")
print(f"Done: {len(locations)} stack(s) processed, {len(errors)} failed.")
pd.DataFrame(errors)

#### 6.1. Location check

`window_lines` × `window_samples` is the multilook window and `window_az_m` × `window_gr_m` its actual ground extent. `line`/`sample` come from the geometry LUT; `line_orbit`/`sample_orbit` from the primary orbit. Differences of more than ~1 pixel mean the window may not be centred on the station.

Note that `dem_height_m` is in the reference of the processing DEM (Copernicus DEM, EGM2008 geoid heights), while the orbit check treats it as ellipsoidal height. A geoid undulation of N metres shifts the orbit-based range by about N·cos θ metres.

In [ ]:
locations

#### 6.2. Per-acquisition table

In [ ]:
acquisitions.head(20)

### 7. Plots for one stack

In [ ]:
if len(locations):
    sel = locations.iloc[0]
    key = (acquisitions["point_id"] == sel["point_id"]) & (acquisitions["stack_id"] == sel["stack_id"]) \
          & (acquisitions["frame"] == sel["frame"])
    a = acquisitions[key]
    p = pairs[(pairs["point_id"] == sel["point_id"]) & (pairs["stack_id"] == sel["stack_id"])
              & (pairs["frame"] == sel["frame"])]
    t = triplets[(triplets["point_id"] == sel["point_id"]) & (triplets["stack_id"] == sel["stack_id"])
                 & (triplets["frame"] == sel["frame"])]
    pol_c = "VV" if "VV" in set(a["pol"]) else a["pol"].iloc[0]

    fig, ax = plt.subplots(1, 3, figsize=(16, 4.5))
    for pol, g in a.groupby("pol"):
        ax[0].plot(g["datetime"], g["sigma0_db"], "o-", label=pol)
    ax[0].set(title=f"σ⁰ ({sel['window_lines']}x{sel['window_samples']} px ≈ "
                    f"{sel['window_az_m']:.0f}x{sel['window_gr_m']:.0f} m multilook)", ylabel="dB")
    ax[0].legend()
    ax[0].tick_params(axis="x", rotation=45)

    n = int(max(p["j"].max(), 0)) + 1 if len(p) else 0
    M = np.eye(n)
    for _, r in p[p["pol"] == pol_c].iterrows():
        M[r["i"], r["j"]] = M[r["j"], r["i"]] = r["coherence"]
    im = ax[1].imshow(M, vmin=0, vmax=1, cmap="viridis")
    ax[1].set(title=f"Coherence {pol_c}", xlabel="acquisition", ylabel="acquisition")
    fig.colorbar(im, ax=ax[1])

    for pol, g in t[t["consecutive"]].groupby("pol"):
        ax[2].plot(g["date_j"], np.degrees(g["closure_phase_rad"]), "o-", label=pol)
    ax[2].axhline(0, color="grey", lw=0.8)
    ax[2].set(title="Closure phase, consecutive triplets", ylabel="deg")
    ax[2].legend()
    ax[2].tick_params(axis="x", rotation=45)

    fig.suptitle(f"{sel['point_id']}  |  {sel['stack_id']}  |  F{sel['frame']}")
    fig.tight_layout()

### 8. Save the results

In [ ]:
locations.to_csv(OUTPUT_DIR / "locations.csv", index=False)
acquisitions.to_csv(OUTPUT_DIR / "acquisitions.csv", index=False)
pairs.to_csv(OUTPUT_DIR / "pairs.csv", index=False)
triplets.to_csv(OUTPUT_DIR / "triplets.csv", index=False)
print(f"Saved to {OUTPUT_DIR.resolve()}")

Outputs in `OUTPUT_DIR`:
- `locations.csv`: one row per station × stack; slant-range position (LUT and orbit), window size in pixels and metres, DEM height.
- `acquisitions.csv`: one row per station × acquisition × polarisation; β⁰/σ⁰ (window mean), LUT values (k_z, incidence, slopes, flattening and SKP phase), orbit geometry and baselines, and the matched near-surface soil moisture (`sm_surface`, `sm_dt_min`) from the availability notebook.
- `pairs.csv`: one row per station × pol × pair; interferometric phase, coherence, temporal and perpendicular baseline.
- `triplets.csv`: one row per station × pol × triplet; closure phase.
- `slc_windows/*.npz`: complex windows `slc[acq, pol, line, sample]` after the phase corrections chosen above, `slc_raw` as stored in the products, the per-pixel `flattening_phase` and `skp_phase` screens `[acq, line, sample]` with the flags of what each product already had applied, plus dates (UTC), product IDs and pixel rows/cols. Other phase choices can be applied later without downloading again.